### load pdf files

In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
   PyMuPDFLoader,
   UnstructuredPDFLoader

)

d:\RAG System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
raw_pdf_text = """Company Financial Report


    The ﬁnancial performance for ﬁscal year 2024
    shows signiﬁcant growth in proﬁtability.
    
    
    
    Revenue increased by 25%.
    
The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10
"""

In [10]:
import re
import html

def clean_text(text):
    
    # 1️⃣ Convert HTML entities
    text = html.unescape(text)

    # 2️⃣ Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # 3️⃣ Fix ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")

    # 4️⃣ Remove special characters except punctuation
    text = re.sub(r"[^\w\s.,!?-]", " ", text)

    # 5️⃣ Remove extra whitespace
    text = " ".join(text.split())

    return text

In [11]:
cleaned_text = clean_text(raw_pdf_text)
print("before cleaning:")
print(raw_pdf_text)
print("after cleaning:")
print(cleaned_text)

before cleaning:
Company Financial Report


    The ﬁnancial performance for ﬁscal year 2024
    shows signiﬁcant growth in proﬁtability.



    Revenue increased by 25%.

The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10

after cleaning:
Company Financial Report The financial performance for fiscal year 2024 shows significant growth in profitability. Revenue increased by 25 . The company s efficiency improved due to workflow optimization. Page 1 of 10


In [12]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
from langchain_core.documents import Document
from typing import List
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],

        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Laod PDF

        loader=PyPDFLoader(pdf_path)
        pages=loader.load()

        ## Process each page

        processed_chunks=[]

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )
            
            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text

    
            


In [18]:
processor=SmartPDFProcessor(chunk_size=500, chunk_overlap=50)
processed_documents = processor.process_pdf("data/pdf/attention.pdf")
for doc in processed_documents:
    print(f"Page: {doc.metadata['page']}, Chars: {doc.metadata['char_count']}")
    print(doc.page_content[:200])
    print("-"*50)

Page: 1, Chars: 2858
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need 
--------------------------------------------------
Page: 1, Chars: 2858
University of Toronto aidan@cs.toronto.edu Łukasz Kaiser ∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are ba
--------------------------------------------------
Page: 1, Chars: 2858
based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more pa
--------------------------------------------------
Page: 1, Chars: 2858
translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of th